# Basic NLP with `text_records()`

This notebook shows a simple handoff from `crategraph` to an NLP library.

The point of using `crategraph` here is that we do not read every file directly. We use the RO-Crate graph to select file entities, keep provenance and basic entity metadata with `text_records()`, then group the NLP results by genre metadata recorded in the graph.

## Install TextBlob

Run this cell once if TextBlob is not already available in your notebook environment.

In [ ]:
!uv pip install textblob

In [ ]:
import re
from pathlib import Path

import pandas as pd
from textblob import TextBlob

from crategraph import Crate

## Load the crate

Load the Australian Corpus of English crate from `data/`.

In [ ]:
crate = Crate(Path("../../data/ldaca/Australian_Corpus_of_English"))
crate.summary()

In [ ]:
crate.glimpse()

## Select files to analyse

Start with file entities, then use crate metadata to group them before doing NLP.

In [ ]:
text_files = crate.select(entity_types=["File"])
text_files.summary()

Keep the file-to-genre part of the graph. This gives us one small graph that contains just the text files and their genre terms.

In [ ]:
files_with_genres = text_files.expand(
    via="ldac:linguisticGenre",
    entity_types=["DefinedTerm"],
)

file_genres = files_with_genres.pattern(
    from_type="File",
    via="ldac:linguisticGenre",
    to_type="DefinedTerm",
)

file_genres.summary()

In [ ]:
genres = sorted(
    file_genres.select(entity_types=["DefinedTerm"]).entities,
    key=lambda entity: entity.name,
)

[genre.name for genre in genres]

## Hand the text to NLP tools

`text_records()` returns one row-like record per text unit, with provenance columns kept alongside the text. Use `include_properties` to carry selected entity metadata into the same rows.

Here we make one file subgraph per genre, sample a few files from each subgraph, and pass those files to `text_records()`.

In [ ]:
def files_for_genre(genre):
    genre_term = file_genres.select(id=genre.id)
    genre_with_files = genre_term.expand(
        via="ldac:linguisticGenre",
        entity_types=["File"],
    )
    return genre_with_files.select(entity_types=["File"])


def sample_entity_ids(graph, limit=15):
    return sorted(entity.id for entity in graph.entities)[:limit]


def text_records_for_genre(genre, limit=15):
    genre_files = files_for_genre(genre)
    entity_ids = sample_entity_ids(genre_files, limit)
    records = genre_files.text_records(
        include_properties=["name"],
        filters={"entity_id": entity_ids},
    )
    return [{**record, "genre": genre.name} for record in records]


rows = []
for genre in genres:
    rows.extend(text_records_for_genre(genre))

records = pd.DataFrame(rows)
records[["entity_id", "genre", "name", "text"]].head()

In [ ]:
word_pattern = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")


def word_count(text):
    return len(word_pattern.findall(text))


records["word_count"] = records["text"].map(word_count)
records["polarity"] = records["text"].map(lambda text: TextBlob(text).sentiment.polarity)

records[["entity_id", "genre", "word_count", "polarity", "name"]].head()

## Summarise by graph-derived genre

The grouping column comes from RO-Crate relationships, not from file names.

In [ ]:
records.groupby("genre").agg(
    documents=("entity_id", "count"),
    mean_words=("word_count", "mean"),
    mean_polarity=("polarity", "mean"),
).sort_values("documents", ascending=False)